# 04. Model Checking di Wumpus World

## Setup

Jalankan sel di bawah ini sekali di awal, sebelum sel mana pun yang lain.

Sel ini memasang dependensi yang diperlukan, mencari folder yang berisi
`logic.py` dan `utils.py`, lalu mengimpornya. Kalau notebook dibuka lewat Google
Colab, repo akan di-clone otomatis. Tidak ada yang perlu diubah di sini.

Environment sudah siap kalau baris terakhir output mencetak
`Check       : tt_entails(P & Q, Q) = True`.

In [ ]:
import importlib.util
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/eycoo/Modul-Praktikum-KK-RKA-25.git"
ON_COLAB = "google.colab" in sys.modules


def ensure_dependencies():
    """Install only the packages this module actually uses."""
    required = {
        "networkx": "networkx",
        "numpy": "numpy",
        "pandas": "pandas",
        "matplotlib": "matplotlib",
        "ipywidgets": "ipywidgets",
        "PIL": "pillow",
        "pygments": "pygments",
    }
    missing = [pkg for mod, pkg in required.items() if importlib.util.find_spec(mod) is None]
    if missing:
        print("Installing:", ", ".join(missing))
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)


def find_environment(start):
    """Locate the folder that holds logic.py and utils.py, searching upward."""
    for root in [start, *start.parents]:
        for candidate in sorted(root.rglob("logic.py")):
            if (candidate.parent / "utils.py").exists():
                return candidate.parent
        if (root / ".git").exists():
            break
    return None


ensure_dependencies()

start_dir = Path.cwd()
if ON_COLAB:
    clone_dir = Path("Modul-Praktikum-KK-RKA-25")
    if not clone_dir.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(clone_dir)], check=True)
    start_dir = clone_dir

ENV_DIR = find_environment(start_dir)
if ENV_DIR is None:
    raise RuntimeError(
        "Environment folder not found. Make sure this notebook is opened from "
        "inside the Modul-Praktikum-KK-RKA-25 repository."
    )
if str(ENV_DIR) not in sys.path:
    sys.path.insert(0, str(ENV_DIR))

import itertools
import warnings

import pandas as pd

# qpsolvers is only used by the SVM code, which this module never touches.
warnings.filterwarnings("ignore", message="no QP solver found")

from logic import *
from notebook import psource
from utils import *

print("Environment :", ENV_DIR)
print("Python      :", sys.version.split()[0])
print("Check       : tt_entails(P & Q, Q) =", tt_entails(expr("P & Q"), expr("Q")))

---
# 4.1 Formalisasi Wumpus World

**Slide 32**

## Penjelasan

Yang perlu masuk:

Simbol yang dipakai, sesuai slide:

- $P_{x,y}$ bernilai benar kalau ada pit di [x, y]
- $W_{x,y}$ bernilai benar kalau ada wumpus di [x, y], hidup maupun mati
- $B_{x,y}$ bernilai benar kalau agent merasakan breeze di [x, y]
- $S_{x,y}$ bernilai benar kalau agent merasakan stench di [x, y]

Lalu kelima aturannya:

- $R_1: \neg P_{1,1}$, tidak ada pit di [1,1]
- $R_2: B_{1,1} \Leftrightarrow (P_{1,2} \lor P_{2,1})$
- $R_3: B_{2,1} \Leftrightarrow (P_{1,1} \lor P_{2,2} \lor P_{3,1})$
- $R_4: \neg B_{1,1}$
- $R_5: B_{2,1}$

Bedakan dua kelompok aturan ini, karena slide juga membedakannya:

- $R_1$ sampai $R_3$ berlaku di **semua** dunia Wumpus. Ini aturan main.
- $R_4$ dan $R_5$ khusus untuk dunia yang sedang dijelajahi agent. Ini percept.

Dua hal yang wajib dibahas:

1. Kenapa aturan breeze pakai bikondisional $\Leftrightarrow$ dan bukan implikasi
   biasa $\Rightarrow$. Kalau cuma satu arah, informasi apa yang hilang.
2. Kenapa aturan breeze harus ditulis untuk **setiap** kotak, dan kenapa di sini
   cuma ditulis untuk kotak yang relevan saja.

## Contoh penerapan

Bangun KB pakai `PropKB`:

- Jelaskan kelas `PropKB` dan atribut `clauses`
- Method `tell`, dan fakta bahwa dia otomatis mengkonversi ke CNF
- Method `ask_generator` dan kenapa dia mengembalikan `{}` atau `None`, bukan
  True atau False
- Method `ask_if_true` sebagai versi yang lebih enak dipakai
- Method `retract`

`tell` satu per satu $R_1$ sampai $R_5$, lalu cetak isi `clauses`.

Bahas hasil konversinya. Bikondisional
$B_{1,1} \Leftrightarrow (P_{1,2} \lor P_{2,1})$ pecah jadi tiga clause. Tunjukkan
langkah pecahnya, karena ini bekal untuk materi CNF minggu depan.

---
# 4.2 Ukuran Ruang Model

**Slide 33**

## Penjelasan

Yang perlu masuk:

- Tujuannya: menentukan apakah $KB \vDash \alpha$ untuk sentence $\alpha$
  tertentu. Contoh dari slide: apakah $\neg P_{1,2}$ entailed oleh KB Wumpus
  World?
- Algoritmanya model checking, yaitu enumerasi semua model lalu cek bahwa
  $\alpha$ benar di setiap model yang membuat KB benar.
- Slide menanyakan dua hal dan sengaja dibiarkan kosong: ada berapa simbol, dan
  ada berapa model yang mungkin. Jawab keduanya di sini.

Jawabannya 7 simbol dan 128 model, tapi jangan langsung ditulis. Minta pembaca
menghitung sendiri dulu, baru dikonfirmasi lewat kode.

## Contoh penerapan

Ambil semua proposition symbol dari KB pakai `prop_symbols()` dan
`associate('&', ...)`. Cetak daftarnya, jumlahnya, dan $2^n$.

Tampilkan juga source code `tt_entails` dan `tt_check_all` pakai `psource()`,
lalu jelaskan alur rekursinya secara singkat. Cukup satu paragraf, jangan
dibedah baris per baris.

Tambahkan satu contoh sederhana yang bukan Wumpus supaya alurnya jelas dulu,
misalnya `tt_entails(P & Q, Q)` yang bernilai True dibanding
`tt_entails(P | Q, Q)` yang bernilai False. Untuk yang kedua, sertakan tabel
4 barisnya dan tunjuk baris mana yang membatalkan entailment.

---
# 4.3 Membangun Ulang Figure 7.9

**Slide 34, Figure 7.9**

## Penjelasan

Yang perlu masuk:

- Figure 7.9 adalah truth table 128 baris dengan kolom $B_{1,1}$, $B_{2,1}$,
  $P_{1,1}$, $P_{1,2}$, $P_{2,1}$, $P_{2,2}$, $P_{3,1}$, lalu $R_1$ sampai $R_5$,
  lalu KB.
- KB bernilai benar hanya ketika $R_1$ sampai $R_5$ semuanya benar, dan itu
  terjadi di tepat 3 dari 128 baris.
- Di ketiga baris itu $P_{1,2}$ selalu salah, jadi tidak ada pit di [1,2].
- Sebaliknya, $P_{2,2}$ kadang benar kadang salah, jadi belum bisa disimpulkan.

Kaitkan dengan Notebook 02 sub-topik 2.3. Kesimpulannya sama persis, tapi di sana
dikerjakan manual dengan 8 model, di sini otomatis dengan 128 model.

## Contoh penerapan

Bangkitkan ulang tabel 128 barisnya, jangan disalin dari buku. Definisikan $R_1$
sampai $R_5$ sebagai `Expr`, loop semua kombinasi 7 simbol pakai
`itertools.product`, evaluasi tiap aturan pakai `pl_true`, lalu tampilkan sebagai
DataFrame.

Verifikasi tiga klaim slide dengan kode:

- Total baris 128
- KB bernilai benar di 3 baris
- Di ketiga baris itu $P_{1,2}$ selalu salah, tapi $P_{2,2}$ tidak

Tampilkan juga ketiga baris itu saja supaya bisa dilihat langsung.

Setelah itu kerjakan hal yang sama lewat `ask_if_true`, lalu tunjukkan hasil yang
mudah disalahpahami: `ask_if_true(~P22)` dan `ask_if_true(P22)` sama-sama
mengembalikan False. Jelaskan bahwa ini bukan bug, dan kaitkan dengan pembahasan
$\alpha_2$ di Notebook 02.

---
# 4.4 Batasan Model Checking

**Slide 36**

## Penjelasan

Yang perlu masuk:

- Rekap singkat apa saja yang sudah dipelajari sepanjang keempat notebook. Bentuk
  tabel dengan kolom konsep dan fungsi atau kelas yang mewakilinya.
- Masalah utama model checking: jumlah model tumbuh $2^n$ terhadap jumlah simbol.
  KB dengan 7 simbol butuh 128 model, tapi Wumpus World ukuran penuh punya
  jauh lebih banyak simbol.
- Pengantar materi minggu depan sesuai slide 36: Propositional Theorem Proving,
  yaitu konstruksi pembuktian tanpa mengkonsultasi model.

Sebutkan apa saja yang akan dibahas minggu depan, tapi jangan dijelaskan di sini:
konversi ke CNF dan resolution, Horn clause dengan forward dan backward chaining,
lalu model checking yang lebih efisien lewat DPLL dan WalkSAT. Semua fungsi itu
sudah ada di `logic.py`, boleh disebut supaya yang penasaran bisa intip duluan.

## Contoh penerapan

Tidak perlu kode baru. Cukup tabel rekap dalam markdown, plus satu perhitungan
kecil yang menunjukkan pertumbuhan $2^n$ untuk beberapa nilai n, misalnya 7, 18,
dan jumlah simbol kalau seluruh 16 kotak grid dimodelkan.

---
# Latihan Soal

Isi bagian ini dengan minimal 3 soal. Soal 2 adalah latihan resmi dari slide,
jadi wajib masuk dan porsinya paling besar.

Format tiap soal: pernyataan soal, cell kosong untuk jawaban, lalu pembahasan
yang dibungkus `<details>`.

## Soal 1

Tingkat pemahaman. Usul arah soal: kenapa `ask_if_true(~P22)` dan
`ask_if_true(P22)` sama-sama mengembalikan False? Apa artinya bagi agent yang
harus memutuskan mau melangkah ke [2,2] atau tidak.

## Soal 2: Latihan 32 Possible Worlds

**Slide 35**

Ini latihan resmi dari slide dan jadi soal utama notebook ini. Soal aslinya:

> Diketahui agent telah sampai pada kondisi seperti pada gambar: tidak ada apa-apa
> pada [1,1], ada input berupa bau (stench) pada [1,2], dan belum tahu isi dari
> [1,3], [2,2], dan [3,1]. Masing-masing dari posisi tersebut dapat berisi sebuah
> lubang (pit) dan maksimal satu posisi dapat berisi monster (Wumpus).
>
> Tentukan kondisi yang mungkin (sebanyak 32 possible worlds).
>
> Tandai kondisi di mana KB bernilai benar dan di mana masing-masing kalimat
> berikut benar:
>
> $\alpha_2$ = "There is no pit in [2,2]"
> $\alpha_3$ = "There is a wumpus in [1,3]"
>
> Dengan demikian tunjukkan bahwa $KB \vDash \alpha_2$ dan $KB \vDash \alpha_3$.

Yang perlu disiapkan sebelum soal ini bisa dikerjakan:

**Simbol dan aturan baru.** KB di sub-topik 4.1 cuma punya pit dan breeze. Untuk
soal ini perlu tambahan simbol $W_{x,y}$ dan $S_{x,y}$, plus aturan stench yang
bentuknya simetris dengan aturan breeze: sebuah kotak berbau jika dan hanya jika
ada wumpus di kotak tetangganya. Aturan ini tidak ada di slide, jadi harus
diturunkan sendiri dari deskripsi sensor di slide 12.

**Kotak mana saja yang perlu diberi aturan.** Minimal [1,1], [1,2], dan [2,1],
karena ketiganya sudah dikunjungi dan percept-nya diketahui.

**Percept lengkap.** Dari gambar: [1,1] tidak ada breeze dan tidak ada stench,
[2,1] ada breeze tanpa stench, [1,2] ada stench tanpa breeze. Yang paling sering
terlewat adalah "tanpa stench" di [2,1], padahal justru itu yang dipakai untuk
menyimpulkan $\alpha_3$.

**Asal angka 32.** Jelaskan perhitungannya: 3 kotak yang bisa berisi pit
memberi $2^3 = 8$, dan wumpus punya 4 kemungkinan posisi yaitu di salah satu
dari tiga kotak itu atau tidak di ketiganya. Totalnya 8 kali 4 sama dengan 32.
Kotak yang sudah dikunjungi tidak divariasikan karena agent masih hidup.

**Bentuk output.** Tampilkan sebagai DataFrame 32 baris dengan kolom untuk pit,
posisi wumpus, KB, $\alpha_2$, dan $\alpha_3$. Lalu tampilkan baris yang KB-nya
benar saja, dan cek entailment-nya.

**Verifikasi silang.** Setelah enumerasi manual, cek ulang lewat `PropKB` dan
`ask_if_true`. Hasilnya harus sama. Catat waktu jalannya pakai `%%time`, karena
KB ini punya banyak simbol dan hasilnya jadi bukti konkret untuk sub-topik 4.4.

## Soal 3

Tingkat analisis. Usul arah soal: selain $\alpha_2$ dan $\alpha_3$, kesimpulan
apa lagi yang bisa ditarik dari KB di Soal 2? Minta pembaca mencari sendiri
minimal dua kesimpulan tambahan lalu membuktikannya dengan kode.

## Soal 4 (opsional)

Ruang untuk soal tambahan. Usul arah: berapa banyak proposition symbol yang
dibutuhkan kalau seluruh grid 4 kali 4 dimodelkan lengkap dengan pit, wumpus,
breeze, dan stench? Berapa jumlah modelnya? Kalau satu model butuh 1 mikrodetik
untuk dievaluasi, berapa lama model checking-nya selesai?

Soal ini menyambung langsung ke materi minggu depan.

Hapus kalau tidak dipakai.